# Spatial Network Routing Optimization V2 (High-Performance Distributed Engine)

This notebook executes a distributed capacity-routing simulation using PySpark to evaluate
large-scale network topologies, generating an optimal activation policy dataset.

**V2 Improvements:**
- Bitboard state representation (integer-based, O(1) operations)
- Transposition table with flag-based caching (eliminates redundant evaluations)
- Pre-computed adjacency lookup tables (zero-allocation hot path)
- Optimized Spark partitioning for compute-bound workloads
- Incremental scoring formulation (enables state-only TT keys)
- Edge encoding: 9 (filled), 0 (empty), 8 (dot), 1/-1 (box owner)

In [ ]:
# Setup & Imports
import numpy as np
import random
import time
import os
import json
import glob
from datetime import datetime, timezone

try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("NetworkOptimizationV2").getOrCreate()
    print("PySpark session initialized. DataFrame API available.")
    try:
        sc = spark.sparkContext
    except Exception:
        print("SparkContext restrito (Unity Catalog Shared Cluster). Modo DataFrame/Pandas UDF.")
        sc = None
except Exception:
    print("PySpark not found. Running in local mode.")
    spark = None
    sc = None

In [ ]:
# High-Performance Core Engine (Bitboard + Transposition Table)
# This code is serialized and sent to Spark workers.

def build_topology_tables(rows, cols):
    """Pre-compute all lookup tables for the bitboard engine."""
    h = 2 * rows + 1
    w = 2 * cols + 1
    edge_to_bit = {}
    bit_to_rc = {}
    bit_to_label = {}
    bit_idx = 0
    for r in range(h):
        for c in range(w):
            if r % 2 == 0 and c % 2 == 1:
                edge_to_bit[(r, c)] = bit_idx
                bit_to_rc[bit_idx] = (r, c)
                bit_to_label[bit_idx] = f"H_{r}_{c}"
                bit_idx += 1
            elif r % 2 == 1 and c % 2 == 0:
                edge_to_bit[(r, c)] = bit_idx
                bit_to_rc[bit_idx] = (r, c)
                bit_to_label[bit_idx] = f"V_{r}_{c}"
                bit_idx += 1
    n_edges = bit_idx
    all_mask = (1 << n_edges) - 1

    box_masks = []
    for r in range(1, h, 2):
        for c in range(1, w, 2):
            top = edge_to_bit[(r - 1, c)]
            bot = edge_to_bit[(r + 1, c)]
            lft = edge_to_bit[(r, c - 1)]
            rgt = edge_to_bit[(r, c + 1)]
            box_masks.append((1 << top) | (1 << bot) | (1 << lft) | (1 << rgt))

    edge_boxes = []
    for b in range(n_edges):
        related = tuple(bm for bm in box_masks if bm & (1 << b))
        edge_boxes.append(related)

    labels = [bit_to_label[i] for i in range(n_edges)]
    label_to_bit = {bit_to_label[i]: i for i in range(n_edges)}

    return n_edges, all_mask, edge_boxes, labels, label_to_bit, bit_to_rc, box_masks


def _closures(edges, edge_bit, edge_boxes):
    new = edges | (1 << edge_bit)
    c = 0
    for bm in edge_boxes[edge_bit]:
        if (new & bm) == bm:
            c += 1
    return c


def _ordered_moves(edges, n_edges, edge_boxes):
    good = []
    normal = []
    for i in range(n_edges):
        if edges & (1 << i):
            continue
        new = edges | (1 << i)
        cl = 0
        for bm in edge_boxes[i]:
            if (new & bm) == bm:
                cl += 1
        if cl > 0:
            good.append((i, cl))
        else:
            normal.append(i)
    return good, normal


def deep_evaluate(edges, depth, alpha, beta, maximizing, n_edges, all_mask, edge_boxes, tt):
    """Recursive evaluation with incremental scoring and transposition table."""
    if depth == 0 or edges == all_mask:
        return 0

    tt_key = (edges, depth, maximizing)
    cached = tt.get(tt_key)
    if cached is not None:
        stored_flag, stored_val = cached
        if stored_flag == 0:
            return stored_val
        elif stored_flag == 1:
            if stored_val >= beta:
                return stored_val
            alpha = max(alpha, stored_val)
        elif stored_flag == 2:
            if stored_val <= alpha:
                return stored_val
            beta = min(beta, stored_val)

    good, normal = _ordered_moves(edges, n_edges, edge_boxes)
    orig_alpha = alpha

    if maximizing:
        best = -10000
        pruned = False
        for move, closed in good:
            child = deep_evaluate(
                edges | (1 << move), depth - 1,
                alpha - closed, beta - closed, True,
                n_edges, all_mask, edge_boxes, tt
            )
            val = closed + child
            if val > best:
                best = val
            alpha = max(alpha, best)
            if beta <= alpha:
                pruned = True
                break
        if not pruned:
            for move in normal:
                child = deep_evaluate(
                    edges | (1 << move), depth - 1,
                    alpha, beta, False,
                    n_edges, all_mask, edge_boxes, tt
                )
                if child > best:
                    best = child
                alpha = max(alpha, best)
                if beta <= alpha:
                    break
    else:
        best = 10000
        pruned = False
        for move, closed in good:
            child = deep_evaluate(
                edges | (1 << move), depth - 1,
                alpha + closed, beta + closed, False,
                n_edges, all_mask, edge_boxes, tt
            )
            val = -closed + child
            if val < best:
                best = val
            beta = min(beta, best)
            if beta <= alpha:
                pruned = True
                break
        if not pruned:
            for move in normal:
                child = deep_evaluate(
                    edges | (1 << move), depth - 1,
                    alpha, beta, True,
                    n_edges, all_mask, edge_boxes, tt
                )
                if child < best:
                    best = child
                beta = min(beta, best)
                if beta <= alpha:
                    break

    if best <= orig_alpha:
        tt[tt_key] = (2, best)
    elif best >= beta:
        tt[tt_key] = (1, best)
    else:
        tt[tt_key] = (0, best)

    return best


def compute_all_scores(edges, depth, n_edges, all_mask, edge_boxes):
    tt = {}
    scores = {}
    for i in range(n_edges):
        if edges & (1 << i):
            continue
        cl = _closures(edges, i, edge_boxes)
        new_edges = edges | (1 << i)
        if cl > 0:
            child = deep_evaluate(
                new_edges, depth - 1, -10001, 10001, True,
                n_edges, all_mask, edge_boxes, tt
            )
            scores[i] = cl + child
        else:
            child = deep_evaluate(
                new_edges, depth - 1, -10001, 10001, False,
                n_edges, all_mask, edge_boxes, tt
            )
            scores[i] = child
    return scores


def get_optimal_configuration(edges, depth, n_edges, all_mask, edge_boxes, labels):
    bit_scores = compute_all_scores(edges, depth, n_edges, all_mask, edge_boxes)
    label_scores = {labels[b]: v for b, v in bit_scores.items()}
    best_val = max(bit_scores.values())
    best_bits = [b for b, v in bit_scores.items() if v == best_val]
    best_label = labels[random.choice(best_bits)]
    return best_label, label_scores


def generate_random_topology(rows, cols, n_edges):
    """Generate a random non-terminal topology as a bitboard integer."""
    total = n_edges
    min_f = int(total * 0.15)
    max_f = int(total * 0.85)
    qty = random.randint(min_f, max_f)
    indices = list(range(n_edges))
    random.shuffle(indices)
    edges = 0
    for i in indices[:qty]:
        edges |= (1 << i)
    return edges


def edges_to_matrix(edges, rows, cols, n_edges, bit_to_rc, box_masks):
    """Convert bitboard to numpy matrix. Edges=9, closed boxes=1, dots=8."""
    h, w = 2 * rows + 1, 2 * cols + 1
    mat = np.zeros((h, w), dtype=np.int8)
    for r in range(0, h, 2):
        for c in range(0, w, 2):
            mat[r, c] = 8
    # All edges are marked as 9 (no player ownership)
    for i in range(n_edges):
        if edges & (1 << i):
            r, c = bit_to_rc[i]
            mat[r, c] = 9
    # Mark closed boxes as 1
    for bm in box_masks:
        if (edges & bm) == bm:
            bits_in_box = []
            for b in range(n_edges):
                if bm & (1 << b):
                    bits_in_box.append(bit_to_rc[b])
            avg_r = sum(rc[0] for rc in bits_in_box) // 4
            avg_c = sum(rc[1] for rc in bits_in_box) // 4
            if avg_r % 2 == 1 and avg_c % 2 == 1:
                mat[avg_r, avg_c] = 1
    return mat

In [ ]:
# Orchestration & Progress Tracking

NUM_SAMPLES = 300000
TAMANHO_LOTE = 5000
DEPTH = 7
ROWS, COLS = 4, 3
SCORE_INDISPONIVEL = -1e9

DIRETORIO_SAIDA = "/Workspace/Users/c092820@corp.caixa.gov.br/CNN"

os.makedirs(DIRETORIO_SAIDA, exist_ok=True)

_n_edges, _all_mask, _edge_boxes, _labels, _label_to_bit, _bit_to_rc, _box_masks = build_topology_tables(ROWS, COLS)
_indice_label = {lab: i for i, lab in enumerate(_labels)}
_n_labels = len(_labels)

print(f"Topology: {ROWS}x{COLS}, {_n_edges} edges, {ROWS*COLS} zones")
print(f"Bitboard range: 0 .. 2^{_n_edges}-1 = {(1 << _n_edges)-1}")


def process_batch_v2(iterator):
    import pandas as pd
    import numpy as np
    import random
    import json

    rows, cols, depth = 4, 3, 7
    n_edges, all_mask, edge_boxes, labels, label_to_bit, bit_to_rc, box_masks = build_topology_tables(rows, cols)

    for pdf in iterator:
        results = []
        for _ in range(len(pdf)):
            while True:
                edges = generate_random_topology(rows, cols, n_edges)
                if edges == all_mask:
                    continue
                try:
                    best_label, scores_dict = get_optimal_configuration(
                        edges, depth, n_edges, all_mask, edge_boxes, labels
                    )
                    mat = edges_to_matrix(edges, rows, cols, n_edges, bit_to_rc, box_masks)
                    results.append({
                        "matriz": [int(x) for x in mat.flatten().tolist()],
                        "best_link": best_label,
                        "scores_dict": json.dumps(scores_dict)
                    })
                    break
                except Exception:
                    pass
        yield pd.DataFrame(results)


def log_progresso(gerados, total, inicio):
    decorrido = time.time() - inicio
    porcentagem = gerados / total * 100
    estimativa = (decorrido / gerados * (total - gerados)) if gerados > 0 else 0
    entrada = {
        "registros_gerados": gerados,
        "total_alvo": total,
        "porcentagem": round(porcentagem, 2),
        "tempo_decorrido_s": round(decorrido, 2),
        "estimativa_restante_s": round(estimativa, 2),
    }
    print(json.dumps(entrada, ensure_ascii=False))


def vetor_scores(scores_dict, indice_label, n_labels):
    vetor = np.full(n_labels, SCORE_INDISPONIVEL, dtype=np.float32)
    for label, valor in scores_dict.items():
        vetor[indice_label[label]] = float(valor)
    return vetor

In [ ]:
# Execution Loop (Distributed Batches) - Optimized Partitioning
import pandas as pd

labels_canonicos = _labels
indice_label = _indice_label
n_labels = _n_labels

print(f"Iniciando V2 de {NUM_SAMPLES} cenarios (Depth {DEPTH})...")
print(f"Destino: {DIRETORIO_SAIDA}")

# ---- CHECKPOINT RECOVERY ----
arquivos_existentes = sorted(glob.glob(os.path.join(DIRETORIO_SAIDA, "dataset_pequeno_*.npz")))
if arquivos_existentes:
    ultimo_arquivo = arquivos_existentes[-1]
    ultimo_lote = int(ultimo_arquivo.split('_')[-1].split('.')[0])
    total_gerado = ultimo_lote * TAMANHO_LOTE
    print(f"Retomando do checkpoint: Lote {ultimo_lote} ({total_gerado} registros).")
else:
    total_gerado = 0
    ultimo_lote = 0

inicio = time.time()

CHUNK_SIZE = 2000
NUM_PARTITIONS = 128

schema = "matriz array<int>, best_link string, scores_dict string"

estados_lote = []
rotulos_lote = []
scores_lote = []
indices_lote = []

while total_gerado < NUM_SAMPLES:
    restante = NUM_SAMPLES - total_gerado
    chunk_atual = min(CHUNK_SIZE, restante)

    if spark:
        df = spark.range(0, chunk_atual, 1, min(NUM_PARTITIONS, chunk_atual))
        res_df = df.mapInPandas(process_batch_v2, schema=schema)
        resultados_chunk = res_df.collect()

        for row in resultados_chunk:
            matriz = np.array(row.matriz, dtype=np.int8).reshape(2 * ROWS + 1, 2 * COLS + 1)
            best_link = row.best_link
            scores_dict = json.loads(row.scores_dict)
            estados_lote.append(matriz)
            rotulos_lote.append(best_link)
            scores_lote.append(vetor_scores(scores_dict, indice_label, n_labels))
            indices_lote.append(total_gerado)
            total_gerado += 1
    else:
        for _ in range(chunk_atual):
            while True:
                edges = generate_random_topology(ROWS, COLS, _n_edges)
                if edges == _all_mask:
                    continue
                try:
                    best_label, scores_dict = get_optimal_configuration(
                        edges, DEPTH, _n_edges, _all_mask, _edge_boxes, _labels
                    )
                    mat = edges_to_matrix(edges, ROWS, COLS, _n_edges, _bit_to_rc, _box_masks)
                    estados_lote.append(mat)
                    rotulos_lote.append(best_label)
                    scores_lote.append(vetor_scores(scores_dict, _indice_label, _n_labels))
                    indices_lote.append(total_gerado)
                    total_gerado += 1
                    break
                except Exception:
                    pass

    log_progresso(total_gerado, NUM_SAMPLES, inicio)

    if len(estados_lote) >= TAMANHO_LOTE or total_gerado >= NUM_SAMPLES:
        if len(estados_lote) > 0:
            ultimo_lote += 1
            caminho_lote = os.path.join(DIRETORIO_SAIDA, f"dataset_pequeno_{ultimo_lote:04d}.npz")
            np.savez_compressed(
                caminho_lote,
                estados=np.array(estados_lote, dtype=np.int8),
                rotulos=np.array(rotulos_lote, dtype=str),
                scores=np.array(scores_lote, dtype=np.float32),
                indices=np.array(indices_lote, dtype=np.int32),
                labels_canonicos=np.array(labels_canonicos, dtype=str),
            )
            print(f"Lote {ultimo_lote:04d} salvo em {caminho_lote}")
            estados_lote = []
            rotulos_lote = []
            scores_lote = []
            indices_lote = []

print(f"Concluido: {total_gerado} registros salvos.")